# Honest Full-Precision Eval — Path B

**Цель**: устранить inference-artifact в сравнении base vs GSPO vs KTO. Все три модели прогоняются через Hugging Face transformers в bf16 с **identical decoding protocol** + **identical Combined Judge** (Cerebras).

**Compute**: RTX 6000 Ada (48GB), ~8.71 units/hour. Estimated: 3 models × 143 calc problems × ~30s = ~3.6h = ~31 units (≪ 600).

**Output**: `evaluation/reports/honest_full_precision_2026-04-30.json` со строгим apple-to-apple сравнением.

**Структура**:
1. Setup (paths, imports, .env)
2. **DECODING_CONFIG** — TODO(human): главное методологическое решение
3. Load eval dataset (143 calc problems)
4. Per-model inference function (base / +GSPO adapter / +KTO adapter)
5. Combined Judge (Cerebras, единый для всех трёх)
6. Run + save

In [ ]:
# Cell 2: Setup (Colab edition)
# ─── GPU check (VRAM-based, robust к названию) ─────────────────
# 9B bf16 = ~22GB weights + KV cache + activations → need ≥24GB VRAM minimum.
# 96GB GPUs (RTX PRO 6000 Blackwell, A100 80GB) дают comfortable headroom для
# co-located scoring (Skywork-PRM-1.5B + tutor 9B).
import subprocess
gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip()
print(f'GPU: {gpu_info}')
gpu_name, memory_str = gpu_info.split(',', 1)
gpu_memory_mib = int(memory_str.strip().split()[0])
assert gpu_memory_mib >= 24_000, (
    f'Insufficient VRAM for 9B bf16: {gpu_memory_mib} MiB. Need >=24GB. '
    f'Switch runtime to A100 / L4 24GB+ / RTX 6000 / RTX PRO 6000 / L40 / A40.'
)
print(f'VRAM check passed: {gpu_memory_mib} MiB ({gpu_memory_mib/1024:.1f} GiB)')

# ─── Mount Drive (for .env + report persistence across sessions) ─────
from google.colab import drive
drive.mount('/content/drive')

# ─── Clone repo + install ─────────────────────────────────────
# NB: GitHub repo is Siesher/MIST (historical typo from MITS — the URL stuck).
import os
if not os.path.exists('/content/MITS'):
    !git clone https://github.com/Siesher/MIST.git /content/MITS
%cd /content/MITS
!git checkout 019-ns-vstar-dpo && git pull origin 019-ns-vstar-dpo
!pip install -q transformers==4.46.0 accelerate==1.1.0 peft==0.13.0 bitsandbytes==0.44.0 openai python-dotenv

# ─── HF auth (for private adapters; if Siesher/mits-qwen3-9b-gspo is public, skip) ─
from huggingface_hub import login as hf_login
# Or set HF_TOKEN in Drive .env — load_dotenv() ниже подхватит автоматически.

# ─── Stdlib + project imports ─────────────────────────────────
import sys, json, time, logging
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List

PROJECT_ROOT = Path('/content/MITS')
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from dotenv import load_dotenv

# Load .env from Drive (keeps Cerebras keys out of the repo + survives Colab restarts)
ENV_PATH = Path('/content/drive/MyDrive/MITS_secrets/.env')
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    print(f'Loaded env from {ENV_PATH}')
    if os.environ.get('HF_TOKEN'):
        hf_login(token=os.environ['HF_TOKEN'])
        print('HF authenticated via .env')
else:
    print(f'No .env at {ENV_PATH}. Cerebras judge will fail — socratic scores=None.\n'
          f'    Create folder MITS_secrets/ on Drive and upload .env with CEREBRAS_API_KEY_1..10.')

from training.scripts.evaluate_stage import (
    SYSTEM_PROMPT_CALC,
    extract_answer,
    check_format_compliance,
    evaluate_combined_quality,
    load_eval_dataset,
)
from training.cerebras_client import CerebrasClient

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('honest_eval')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f'Device: {DEVICE} | bf16: {torch.cuda.is_bf16_supported()}')

## Cell 3 — Decoding Configuration (зафиксирован)

Параметры применяются ОДИНАКОВО ко всем трём моделям (base, GSPO, KTO). Зафиксировано:

| Параметр | Значение | Обоснование |
|----------|----------|-------------|
| `num_predict` | 4096 | Покрывает 95-percentile thinking длин (GSPO учился с budget=2048; 4096 даёт запас на hard problems без overhead 8192). |
| `enable_thinking` | `True` | Матчит training distribution GSPO/KTO + native режим Qwen3.5-9B. False нивелировал бы RL-effect целиком — unfair. |
| `temperature` | 0.0 | Greedy для deterministic accuracy. Diversity sampling — Phase 1 (V-STaR). |
| `system_prompt` | `SYSTEM_PROMPT_CALC` | Apple-to-apple с предыдущими compare_base_vs_gspo отчётами. |

**Что это даёт для диплома**: section "Methodology — inference protocol" в одну таблицу. Альтернатива (запустить второй раз с `enable_thinking=False`) — рассматривается как _Appendix-grade ablation_, если останется compute после Phase 1-3.

In [ ]:
# Cell 4: Decoding config (filled — single-protocol run)
# Same config applied to all three models (base, GSPO, KTO).

DECODING_CONFIG = {
    # 4096: covers 95-percentile thinking (GSPO trained budget=2048, eval slightly
    # larger to avoid truncation on hard problems). 8192 is overkill for 9B+thinking.
    'num_predict': 4096,
    # True: matches GSPO/KTO training distribution (chat_template_kwargs.enable_thinking
    # =True in grpo_qwen3_5_9b_(8).ipynb). Qwen3.5-9B base also natively supports <think>.
    # False would nullify the RL effect — unfair to GSPO/KTO. Keep one consistent mode.
    'enable_thinking': True,
    # 0.0: greedy decoding for accuracy eval. Diversity sampling is for V-STaR (Phase 1).
    'temperature': 0.0,
    # Same calc system prompt used in compare_base_vs_gspo_*.json — preserves apple-to-
    # apple with prior reports for the Cerebras-only path; only inference layer changes.
    'system_prompt': SYSTEM_PROMPT_CALC,
    'rationale': (
        'Single thinking-on protocol mirrors training distribution + production deployment. '
        '4096 budget eliminates truncation. Greedy decoding for deterministic accuracy.'
    ),
}

assert all(v is not None for v in DECODING_CONFIG.values()), 'Fill in DECODING_CONFIG keys'
logger.info(f'Decoding config: {DECODING_CONFIG}')

In [ ]:
# Cell 5: Load eval dataset — calc subset only (numeric / latex_boxed)
EVAL_PATH = PROJECT_ROOT / 'training/data/eval_dataset.jsonl'
all_problems = load_eval_dataset(str(EVAL_PATH))
calc_problems = [p for p in all_problems if p.get('answer_type') in ('numeric', 'latex_boxed')]
logger.info(f'Total: {len(all_problems)} | Calc subset: {len(calc_problems)}')
# Sanity
from collections import Counter
logger.info(f'By domain: {Counter(p["domain"] for p in calc_problems)}')
logger.info(f'By difficulty: {Counter(p["difficulty"] for p in calc_problems)}')

In [ ]:
# Cell 6: Model loading + inference
BASE_MODEL_ID = 'Qwen/Qwen3.5-9B'  # matches BASE_MODEL in grpo_qwen3_5_9b_(8).ipynb
ADAPTERS = {
    'base': None,
    'gspo': 'Siesher/mits-qwen3-9b-gspo',
    'kto':  'Siesher/mits-qwen3-9b-kto',
}

def load_model_with_adapter(adapter_id: str | None):
    """Load Qwen3.5-9B base in bf16, optionally apply LoRA adapter and merge."""
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
    )
    if adapter_id:
        model = PeftModel.from_pretrained(model, adapter_id)
        model = model.merge_and_unload()
        logger.info(f'Merged adapter {adapter_id}')
    model.eval()
    return model, tokenizer

def generate_one(model, tokenizer, prompt: str) -> str:
    messages = [
        {'role': 'system', 'content': DECODING_CONFIG['system_prompt']},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=DECODING_CONFIG['enable_thinking'],
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=DECODING_CONFIG['num_predict'],
            do_sample=DECODING_CONFIG['temperature'] > 0,
            temperature=max(DECODING_CONFIG['temperature'], 1e-5),
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return completion

In [ ]:
# Cell 7: Eval loop for one stage
def eval_stage(stage_name: str, adapter_id: str | None, problems: List[Dict]) -> Dict[str, Any]:
    logger.info(f'=== Stage: {stage_name} ({adapter_id or "base"}) ===')
    model, tokenizer = load_model_with_adapter(adapter_id)
    cerebras = CerebrasClient()
    completions = []
    t0 = time.time()
    for i, p in enumerate(problems):
        completion = generate_one(model, tokenizer, p['prompt'])
        extracted = extract_answer(completion)
        fmt = check_format_compliance(completion)
        visible = completion.split('</think>')[-1].strip() if '</think>' in completion else completion
        judge = evaluate_combined_quality(p['prompt'], visible, p['ground_truth'], cerebras)
        completions.append({
            'idx': i, 'domain': p['domain'], 'difficulty': p['difficulty'],
            'truth': p['ground_truth'], 'extracted': extracted,
            'correct': judge.get('is_correct'),
            'socratic_score': judge.get('socratic_score'),
            'no_answer_leak': judge.get('no_answer_leak'),
            **fmt,
        })
        if (i + 1) % 10 == 0:
            elapsed = time.time() - t0
            acc = sum(1 for c in completions if c['correct']) / len(completions)
            logger.info(f'  {i+1}/{len(problems)} | acc={acc:.3f} | {elapsed/60:.1f} min')
    # Free GPU
    del model; torch.cuda.empty_cache()
    accuracy = sum(1 for c in completions if c['correct']) / len(completions)
    return {
        'stage': stage_name, 'adapter': adapter_id, 'n': len(completions),
        'accuracy': accuracy,
        'avg_socratic': sum(c.get('socratic_score') or 0 for c in completions) / len(completions),
        'leak_rate': sum(1 for c in completions if (c.get('no_answer_leak') or 2) < 2) / len(completions),
        'completions': completions,
    }

In [ ]:
# Cell 8: Run all three stages and save report (Colab — copy to Drive too)
results = {}
for stage_name, adapter_id in ADAPTERS.items():
    results[stage_name] = eval_stage(stage_name, adapter_id, calc_problems)

report = {
    'protocol': 'honest_full_precision',
    'timestamp': datetime.utcnow().isoformat(),
    'decoding_config': {k: v for k, v in DECODING_CONFIG.items() if k != 'system_prompt'},
    'system_prompt_hash': hash(DECODING_CONFIG['system_prompt']),
    'n_problems': len(calc_problems),
    'base': {k: v for k, v in results['base'].items() if k != 'completions'},
    'gspo': {k: v for k, v in results['gspo'].items() if k != 'completions'},
    'kto':  {k: v for k, v in results['kto'].items()  if k != 'completions'},
    'completions': {stage: r['completions'] for stage, r in results.items()},
}

# Save to repo (so it can be committed) + Drive mirror (so it survives Colab teardown)
date_tag = datetime.utcnow().strftime('%Y%m%d')
out_repo  = PROJECT_ROOT / f'evaluation/reports/honest_full_precision_{date_tag}.json'
out_drive = Path(f'/content/drive/MyDrive/MITS_secrets/honest_full_precision_{date_tag}.json')
out_repo.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
if out_drive.parent.exists():
    out_drive.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    logger.info(f'Saved (Drive mirror): {out_drive}')
logger.info(f'Saved (repo): {out_repo}')

print('\n=== Honest comparison (full-precision bf16, identical decoding) ===')
for stage in ['base', 'gspo', 'kto']:
    r = results[stage]
    print(f"{stage:5s}  acc={r['accuracy']:.3f}  socratic={r['avg_socratic']:.3f}  leak_rate={r['leak_rate']:.3f}")